> **Note on this notebook — why it carries no outputs**
>
> This notebook does not train a model. It was written to load a checkpoint produced
> by an earlier Kaggle GPU session and evaluate it, which is why the cell below looks
> for an existing `.ckpt` rather than calling `trainer.fit`. That session was not
> persistent and the checkpoint was lost with it, so the notebook cannot be re-executed
> as written and is included as a record of what was done rather than as a run.
>
> **What it reported:** the Temporal Fusion Transformer with its original, untuned
> settings, at a test MAE of 1.53 degrees. That checkpoint was trained with a learning
> rate of 0.0003. The model construction cell below specifies 0.001, so the code here
> would not reproduce 1.53 even if it were retrained; the two were separated when the
> configuration was changed between sessions.
>
> **How the figure is used:** 1.53 appears in the dissertation only as a description of
> the earlier comparison, the one that appeared to favour the recurrent network before
> every model was given an equivalent hyperparameter search. It is not offered as a
> matched-protocol measurement and no reported conclusion rests on it. The comparison
> that replaces it is in `17_hyperparameter_search.ipynb` and `19_final_results.ipynb`,
> both of which are executed.
>
> Retraining at 0.001 and presenting the result as the untuned figure would have changed
> the number the report describes, and loading the tuned checkpoint would have reported
> the tuned model under an untuned heading. Neither would have been honest, so the
> notebook is left unexecuted and explained instead.

# Final Year Project — Weather Forecasting
## Notebook 3 (Kaggle): Temporal Fusion Transformer (TFT)
**Student:** Tharun Bisai  
**Goal:** Train TFT on ALL 5 cities simultaneously with built-in XAI (variable importance + attention)  
**Comparison target:** Beat LSTM baseline MAE 1.39°C, R² 0.9452

---
### Kaggle setup (one-time, before running)
1. Upload all 5 city CSVs from `D:\Final year project\Dataset\` as a **Private Kaggle Dataset**  
   Suggested title: **`multi-city-weather-2000-2009`**
2. In this notebook's right-side panel:
   - **Accelerator** → `GPU T4 × 2`  (or P100 — either works)
   - **Internet** → `On`  (needed for `pip install`)
   - **Add Data** → attach your `multi-city-weather-2000-2009` dataset
3. Run all cells in order. Outputs save to `/kaggle/working/` and persist as notebook artifacts.

**No Google Drive. No 90-minute disconnects. No auth popups.** 🚀

## 1. Install Libraries (Kaggle)

In [ ]:
# Pin versions to match Lightning AI training environment
# pandas pinned to fix StringDtype pickle compatibility on checkpoint load
!pip install -q pytorch-lightning==2.6.4 pytorch-forecasting==1.7.0 pandas==2.3.0
print('Libraries installed - RESTART KERNEL once then Run All')


## 2. Setup Paths (Kaggle environment)

In [ ]:
import os

# Kaggle conventions:
#   /kaggle/input/  → read-only (all attached datasets live here)
#   /kaggle/working/ → writable, persists as notebook output

# ── Auto-discover CSV files anywhere under /kaggle/input/ ─────────────────────
def find_csv_dir(root='/kaggle/input'):
    """Walk the full input tree and return the folder containing the weather CSVs."""
    target = 'jena_germany_2000_2009_hourly.csv'
    for dirpath, dirnames, filenames in os.walk(root):
        if target in filenames:
            return dirpath + '/'
    return None

DATA_DIR = find_csv_dir()

if DATA_DIR is None:
    # Print the full tree to help debug
    print('❌ Could not find CSV files. Full /kaggle/input/ structure:')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        print('  ' * level + os.path.basename(root) + '/')
        for f in files:
            print('  ' * (level+1) + f)
    raise FileNotFoundError('Weather CSVs not found — check dataset is attached correctly.')

# Writable output dirs
WORK = '/kaggle/working/'
os.makedirs(WORK + 'models',    exist_ok=True)
os.makedirs(WORK + 'plots',     exist_ok=True)
os.makedirs(WORK + 'processed', exist_ok=True)

print(f'✅ DATA_DIR (read-only): {DATA_DIR}')
print(f'✅ WORK     (writable):  {WORK}')
print(f'Files found: {os.listdir(DATA_DIR)}')

## 3. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, Baseline
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE, RMSE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# PyTorch 2.6+ fix: allow GroupNormalizer when loading checkpoint
torch.serialization.add_safe_globals([GroupNormalizer])

torch.manual_seed(42)
np.random.seed(42)
pl.seed_everything(42)

DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print('  GPU', i, ':', torch.cuda.get_device_name(i))
print('PyTorch:', torch.__version__)
print('lightning.pytorch:', pl.__version__)


## 4. Build / Load Master Dataset (All 5 Cities)

In [ ]:
# ── Build master_all_cities.csv from raw CSVs (Kaggle: always rebuild fresh) ──
import re

PROCESSED = WORK + 'processed/master_all_cities.csv'

city_meta = {
    'jena_germany_2000_2009_hourly.csv':    ('Jena',     50.9227, 11.5865),
    'london_uk_2000_2009_hourly.csv':       ('London',   51.5074, -0.1278),
    'newyork_usa_2000_2009_hourly.csv':     ('New York', 40.7128, -74.0060),
    'sydney_australia_2000_2009_hourly.csv':('Sydney',  -33.8688, 151.2093),
    'tokyo_japan_2000_2009_hourly.csv':     ('Tokyo',    35.6762, 139.6503),
}

rename_map = {
    'temperature_2m':                       'temperature',
    'relative_humidity_2m':                 'humidity',
    'dew_point_2m':                         'dew_point',
    'precipitation':                        'precipitation',
    'rain':                                 'rain',
    'wind_speed_10m':                       'wind_speed',
    'wind_direction_10m':                   'wind_direction',
    'wind_gusts_10m':                       'wind_gusts',
    'pressure_msl':                         'pressure_msl',
    'surface_pressure':                     'surface_pressure',
    'cloud_cover':                          'cloud_cover',
    'cloud_cover_low':                      'cloud_cover_low',
    'cloud_cover_mid':                      'cloud_cover_mid',
    'cloud_cover_high':                     'cloud_cover_high',
    'shortwave_radiation':                  'shortwave_radiation',
    'direct_radiation':                     'direct_radiation',
    'vapour_pressure_deficit':              'vapour_pressure_deficit',
    'wet_bulb_temperature_2m':              'wet_bulb_temp',
    'total_column_integrated_water_vapour': 'water_vapour',
    'soil_temperature_0_to_7cm':            'soil_temperature',
    'et0_fao_evapotranspiration':           'evapotranspiration',
}

def normalize_col(col: str) -> str:
    """Strip units like ' (°C)', ' (%)', etc."""
    col = re.sub(r'\s*\([^)]*\)', '', col)
    return col.strip()

print('Building master_all_cities.csv from raw CSVs...')
frames = []
for fname, (city, lat, lon) in city_meta.items():
    fpath = DATA_DIR + fname
    if not os.path.exists(fpath):
        # Try without subdirectory in case files are flat
        alt = os.path.join(DATA_DIR, fname)
        if not os.path.exists(alt):
            raise FileNotFoundError(f'{fpath} — check dataset attachment')
        fpath = alt

    print(f'  Loading {city}...', end=' ')
    d = pd.read_csv(fpath, skiprows=3, parse_dates=['time'])

    # 1. Normalize columns: strip ' (°C)', ' (%)', etc.
    d.columns = [normalize_col(c) for c in d.columns]

    # 2. Drop pandas-auto-renamed dupes (cloud_cover.1, .2, .3)
    d = d.loc[:, ~d.columns.str.contains(r'\.')]

    # 3. Drop duplicates after normalization (keep first)
    d = d.loc[:, ~d.columns.duplicated()]

    # 4. Apply rename map
    d = d.rename(columns=rename_map)

    # 5. Keep only known clean columns + time
    keep = ['time'] + [c for c in rename_map.values() if c in d.columns]
    d = d[keep]

    if 'temperature' not in d.columns:
        raise KeyError(f'{city}: NO temperature column. Original: {pd.read_csv(fpath, skiprows=3, nrows=0).columns.tolist()}')

    d['city'] = city
    d['lat']  = lat
    d['lon']  = lon

    # Cyclical time features
    d['hour']      = d['time'].dt.hour
    d['month']     = d['time'].dt.month
    d['dayofyear'] = d['time'].dt.dayofyear
    d['hour_sin']      = np.sin(2 * np.pi * d['hour']      / 24)
    d['hour_cos']      = np.cos(2 * np.pi * d['hour']      / 24)
    d['month_sin']     = np.sin(2 * np.pi * d['month']     / 12)
    d['month_cos']     = np.cos(2 * np.pi * d['month']     / 12)
    d['dayofyear_sin'] = np.sin(2 * np.pi * d['dayofyear'] / 365)
    d['dayofyear_cos'] = np.cos(2 * np.pi * d['dayofyear'] / 365)

    # Wind U/V components
    if 'wind_speed' in d.columns and 'wind_direction' in d.columns:
        wd_rad = np.deg2rad(d['wind_direction'])
        d['wind_u'] = -d['wind_speed'] * np.sin(wd_rad)
        d['wind_v'] = -d['wind_speed'] * np.cos(wd_rad)

    # Lag features
    for lag in [1, 3, 6, 24]:
        d[f'temp_lag_{lag}h'] = d['temperature'].shift(lag)

    # Rolling averages
    for win in [6, 24]:
        d[f'temp_roll_{win}h'] = d['temperature'].shift(1).rolling(win).mean()

    frames.append(d)
    print(f'{len(d):,} rows ✓ ({len(d.columns)} cols)')

master = pd.concat(frames, ignore_index=True)
master = master.sort_values(['city', 'time']).reset_index(drop=True)
master.to_csv(PROCESSED, index=False)
print(f'\n✅ Saved → {PROCESSED}')
print(f'Total rows: {len(master):,}')
print(f'Columns:    {len(master.columns)}')
print(f'Cities:     {master["city"].unique().tolist()}')
print(f'Date range: {master["time"].min()} → {master["time"].max()}')
master.head()

## 5. Prepare Data for TFT (NaN cleanup + time_idx)

In [ ]:
# TFT needs a unique integer 'time_idx' PER GROUP (per city)
df = master.copy()
df = df.sort_values(['city', 'time']).reset_index(drop=True)

# Drop pandas-auto-renamed duplicate columns (safety)
dot_cols = [c for c in df.columns if '.' in c]
if dot_cols:
    print(f'Dropping duplicate columns: {dot_cols}')
    df = df.drop(columns=dot_cols)

# ── NaN diagnostics + cleanup ─────────────────────────────────────────────────
print('\nNaN counts BEFORE cleanup:')
print(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False).head(15))

# CRITICAL: temperature must be non-NaN. Forward-fill within each city.
df['temperature'] = df.groupby('city')['temperature'].transform(
    lambda x: x.ffill().bfill()
)

# For other numeric columns: forward-fill within city, then 0 for any remainder
fill_cols = [c for c in df.columns
             if c not in ['time', 'city', 'lat', 'lon', 'time_idx']
             and df[c].dtype != 'O']
for col in fill_cols:
    df[col] = df.groupby('city')[col].transform(lambda x: x.ffill().bfill())
    df[col] = df[col].fillna(0)

# Create time_idx within each city (0, 1, 2, ...)
df['time_idx'] = df.groupby('city').cumcount()

# Final dropna for any rows still missing critical columns
df = df.dropna(subset=['temperature', 'city', 'lat', 'lon']).reset_index(drop=True)

# Sanity checks
assert df['temperature'].isna().sum() == 0, 'temperature still has NaN!'
assert np.isfinite(df['temperature']).all(), 'temperature has inf!'

print(f'\nNaN counts AFTER cleanup:')
remaining = df.isna().sum()[df.isna().sum() > 0]
if len(remaining) == 0:
    print('  ✓ NO NaN remaining anywhere')
else:
    print(remaining)

print(f'\nFinal dataset: {len(df):,} rows × {len(df.columns)} columns')
print(f'\nPer-city counts:')
print(df.groupby('city')['time_idx'].agg(['count','min','max']))

## 6. Define Features for TFT

In [ ]:
# Static features (fixed per city)
static_categoricals = ['city']
static_reals        = ['lat', 'lon']

# Time-varying KNOWN features (calendar — we always know future values)
time_varying_known_reals = [
    'time_idx',
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dayofyear_sin', 'dayofyear_cos',
]
time_varying_known_categoricals = []

# Candidate UNKNOWN features (only known up to the present)
candidate_unknowns = [
    'temperature',                    # target also goes here (TFT auto-handles)
    'humidity', 'dew_point',
    'precipitation', 'rain',
    'wind_speed', 'wind_direction', 'wind_gusts',
    'wind_u', 'wind_v',
    'pressure_msl', 'surface_pressure',
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'shortwave_radiation', 'direct_radiation',
    'vapour_pressure_deficit', 'wet_bulb_temp',
    'water_vapour', 'soil_temperature',
    'evapotranspiration',
]

# Only keep features actually present in df
time_varying_unknown_reals = [c for c in candidate_unknowns if c in df.columns]
missing = [c for c in candidate_unknowns if c not in df.columns]

TARGET = 'temperature'

print(f'Static categorical: {static_categoricals}')
print(f'Static real:        {static_reals}')
print(f'Known reals:        {len(time_varying_known_reals)} features')
print(f'Unknown reals:      {len(time_varying_unknown_reals)} features')
print(f'  → {time_varying_unknown_reals}')
if missing:
    print(f'Missing (skipped):  {missing}')
print(f'Target:             {TARGET}')

## 7. Train / Validation / Test Split

In [ ]:
# Time-based split (no leakage)
MAX_TIME_IDX = df['time_idx'].max()

TRAIN_END = int(MAX_TIME_IDX * 0.70)
VAL_END   = int(MAX_TIME_IDX * 0.85)

ENCODER_LEN = 168   # 7 days lookback
DECODER_LEN = 24    # predict next 24 hours

print(f'Time index range: 0 → {MAX_TIME_IDX:,}')
print(f'Train: 0 → {TRAIN_END:,}')
print(f'Val:   {TRAIN_END:,} → {VAL_END:,}')
print(f'Test:  {VAL_END:,} → {MAX_TIME_IDX:,}')
print(f'Encoder length (lookback): {ENCODER_LEN} hours')
print(f'Decoder length (forecast): {DECODER_LEN} hours')

## 8. Create TimeSeriesDataSet

In [ ]:
training = TimeSeriesDataSet(
    df[df['time_idx'] <= TRAIN_END],
    time_idx                       = 'time_idx',
    target                         = TARGET,
    group_ids                      = ['city'],
    min_encoder_length             = ENCODER_LEN // 2,
    max_encoder_length             = ENCODER_LEN,
    min_prediction_length          = 1,
    max_prediction_length          = DECODER_LEN,
    static_categoricals            = static_categoricals,
    static_reals                   = static_reals,
    time_varying_known_reals       = time_varying_known_reals,
    time_varying_known_categoricals= time_varying_known_categoricals,
    time_varying_unknown_reals     = time_varying_unknown_reals,
    # ⚠️ NO 'softplus' — temperature can be negative (Jena/NY winter)
    target_normalizer              = GroupNormalizer(groups=['city']),
    add_relative_time_idx          = True,
    add_target_scales              = True,
    add_encoder_length             = True,
    allow_missing_timesteps        = True,
)

validation = TimeSeriesDataSet.from_dataset(
    training, df[df['time_idx'] <= VAL_END],
    predict=False, stop_randomization=True
)

testing = TimeSeriesDataSet.from_dataset(
    training, df, predict=False, stop_randomization=True
)

BATCH = 128   # Kaggle T4×2 can handle bigger batches than Colab T4
train_loader = training.to_dataloader(train=True,  batch_size=BATCH, num_workers=2, persistent_workers=True)
val_loader   = validation.to_dataloader(train=False, batch_size=BATCH, num_workers=2, persistent_workers=True)
test_loader  = testing.to_dataloader(train=False, batch_size=BATCH, num_workers=2, persistent_workers=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

## 9. Sanity Check — Baseline (Last Value)

In [ ]:
# A naive baseline that just predicts the last observed value
baseline = Baseline()
baseline_preds = baseline.predict(val_loader, return_y=True)

# Move tensors to CPU before metric computation (avoids device mismatch)
baseline_output = baseline_preds.output.cpu() if hasattr(baseline_preds.output, 'cpu') else baseline_preds.output
baseline_y = baseline_preds.y[0].cpu() if isinstance(baseline_preds.y, tuple) else baseline_preds.y.cpu()

# Compute MAE manually with numpy (always works regardless of device)
baseline_mae = float(np.mean(np.abs(baseline_output.numpy() - baseline_y.numpy())))

print(f'Naive baseline MAE (val set): {baseline_mae:.4f} °C')
print('TFT must beat this!')

## 10. Build TFT Model

In [ ]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate              = 1e-3,
    hidden_size                = 64,
    attention_head_size        = 4,
    dropout                    = 0.2,
    hidden_continuous_size     = 32,
    output_size                = 7,                  # 7 quantile outputs
    loss                       = QuantileLoss(),
    log_interval               = 50,
    reduce_on_plateau_patience = 4,
    optimizer                  = 'adamw',
)

n_params = sum(p.numel() for p in tft.parameters())
print(f'TFT parameters: {n_params:,}')
print(f'\nQuantiles predicted: 7 levels (10%, 25%, 50%, 75%, 90% + edges)')
print('This gives us probabilistic forecasts with uncertainty bands!')

## 11. Train TFT

In [ ]:
import glob

# Locate checkpoint from attached dataset
ckpt_candidates = (
    glob.glob('/kaggle/input/*/last.ckpt') +
    glob.glob('/kaggle/input/*/*.ckpt')    +
    glob.glob('/kaggle/input/*/**/*.ckpt', recursive=True) +
    glob.glob(WORK + 'models/*.ckpt')
)
seen = set()
ckpt_candidates = [c for c in ckpt_candidates if not (c in seen or seen.add(c))]

if not ckpt_candidates:
    raise FileNotFoundError('No checkpoint found in /kaggle/input/*')

RESUME_CKPT = sorted(ckpt_candidates, key=os.path.getmtime)[-1]
print('Checkpoint found:', RESUME_CKPT)
print('Skipping training - using checkpoint as-is (29/30 epochs already trained)')
print('Marginal improvement from final epoch is typically <1% MAE.')

# Create empty ckpt placeholder so downstream cells expecting  still work
class _CkptStub:
    best_model_path = ''
ckpt = _CkptStub()


## 12. Evaluate on Test Set (All Cities)

In [ ]:
# Load model weights directly from checkpoint (skip training, use as-is)
best_path = ckpt.best_model_path or RESUME_CKPT
print('Loading model from:', best_path)

import torch
from pytorch_forecasting.data import GroupNormalizer
torch.serialization.add_safe_globals([GroupNormalizer])

best_tft = TemporalFusionTransformer.load_from_checkpoint(
    best_path,
    weights_only=False,
    map_location=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)
print('Model loaded')

# Predict on test set
predictions = best_tft.predict(test_loader, return_y=True, return_x=True, mode='prediction')
preds   = predictions.output.cpu().numpy()
actuals = predictions.y[0].cpu().numpy()

mae  = mean_absolute_error(actuals.flatten(), preds.flatten())
rmse = np.sqrt(mean_squared_error(actuals.flatten(), preds.flatten()))
r2   = r2_score(actuals.flatten(), preds.flatten())

print('=' * 50)
print('TFT - OVERALL TEST RESULTS (ALL 5 CITIES)')
print('=' * 50)
print('MAE: ', round(mae, 4), 'C')
print('RMSE:', round(rmse, 4), 'C')
print('R2:  ', round(r2, 4))
print('=' * 50)
print('LSTM baseline MAE: 1.3928 C')
print('TFT MAE:', round(mae, 4), 'C', '(BETTER)' if mae < 1.3928 else '(worse)')


## 13. Per-City Breakdown

In [ ]:
# Get city for each prediction
city_indices = predictions.x['groups'].cpu().numpy().flatten()

results_per_city = []
city_list = sorted(df['city'].unique())
for c_idx, city in enumerate(city_list):
    mask = city_indices == c_idx
    if mask.sum() == 0: continue
    city_mae  = mean_absolute_error(actuals[mask].flatten(), preds[mask].flatten())
    city_rmse = np.sqrt(mean_squared_error(actuals[mask].flatten(), preds[mask].flatten()))
    city_r2   = r2_score(actuals[mask].flatten(), preds[mask].flatten())
    results_per_city.append({
        'City': city, 'MAE': city_mae, 'RMSE': city_rmse, 'R²': city_r2, 'Samples': int(mask.sum())
    })

city_df = pd.DataFrame(results_per_city)
print('=' * 60)
print('PER-CITY TEST PERFORMANCE')
print('=' * 60)
print(city_df.to_string(index=False))
city_df.to_csv(WORK + 'processed/tft_per_city.csv', index=False)

## 14. Variable Importance (XAI — Built In!)

In [ ]:
# This is a HUGE win — TFT gives feature importance for free
raw_predictions = best_tft.predict(val_loader, mode='raw', return_x=True)
interpretation = best_tft.interpret_output(raw_predictions.output, reduction='sum')

fig = best_tft.plot_interpretation(interpretation)
for i, axes in enumerate(fig if isinstance(fig, list) else [fig]):
    plt.savefig(WORK + f'plots/09_tft_importance_{i}.png', dpi=150, bbox_inches='tight')
plt.show()

print('Plots saved! Look at:')
print('  - Encoder variable importance (which past features matter)')
print('  - Decoder variable importance (which known-future features matter)')
print('  - Static variable importance (which city/lat/lon features matter)')
print('  - Attention over time (which past hours TFT focused on)')

## 15. Probabilistic Forecast Visualisation

In [ ]:
# TFT outputs quantiles — plot uncertainty bands
quantile_preds = best_tft.predict(test_loader, mode='quantiles', return_y=True, return_x=True)
q_preds = quantile_preds.output.cpu().numpy()   # (samples, horizon, quantiles)
y_true  = quantile_preds.y[0].cpu().numpy()

n_plot = 6
fig, axes = plt.subplots(n_plot, 1, figsize=(14, 3*n_plot))
for i in range(n_plot):
    ax = axes[i]
    horizon = np.arange(DECODER_LEN)
    ax.fill_between(horizon, q_preds[i, :, 0], q_preds[i, :, -1],
                    alpha=0.20, color='#2563eb', label='10–90% range')
    ax.fill_between(horizon, q_preds[i, :, 1], q_preds[i, :, -2],
                    alpha=0.35, color='#2563eb', label='25–75% range')
    ax.plot(horizon, q_preds[i, :, 3], color='#1e40af', lw=2, label='Median (P50)')
    ax.plot(horizon, y_true[i], color='#dc2626', lw=2, ls='--', label='Actual')
    ax.set_title(f'Sample {i+1} — 24-hour Probabilistic Forecast')
    ax.set_xlabel('Hours ahead')
    ax.set_ylabel('Temperature (°C)')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(WORK + 'plots/10_tft_probabilistic_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 16. MAE by Forecast Horizon

In [ ]:
horizon_mae_tft = [mean_absolute_error(actuals[:, h], preds[:, h]) for h in range(DECODER_LEN)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, DECODER_LEN+1), horizon_mae_tft, marker='o', color='#2563eb',
        linewidth=2, label='TFT')
ax.axhline(y=mae, color='#16a34a', linestyle='--', linewidth=1.5,
           label=f'TFT Avg MAE: {mae:.3f}°C')
ax.axhline(y=1.3928, color='#dc2626', linestyle=':', linewidth=1.5,
           label='LSTM Baseline (1.39°C)')
ax.set_xlabel('Forecast Horizon (hours ahead)')
ax.set_ylabel('MAE (°C)')
ax.set_title('TFT MAE by Forecast Horizon (vs LSTM Baseline)', fontweight='bold')
ax.set_xticks(range(1, DECODER_LEN+1))
ax.legend()
plt.tight_layout()
plt.savefig(WORK + 'plots/11_tft_horizon_mae.png', dpi=150, bbox_inches='tight')
plt.show()

## 17. Save Results

In [ ]:
# Append TFT results to model_results.csv
results_path = WORK + 'processed/model_results.csv'
tft_result = {
    'model': 'TFT',
    'city':  'All 5',
    'MAE':   round(mae, 4),
    'RMSE':  round(rmse, 4),
    'R2':    round(r2, 4),
    'seq_len':    ENCODER_LEN,
    'pred_steps': DECODER_LEN,
    'features':   len(time_varying_unknown_reals),
}

if os.path.exists(results_path):
    res_df = pd.read_csv(results_path)
    res_df = pd.concat([res_df, pd.DataFrame([tft_result])], ignore_index=True)
else:
    res_df = pd.DataFrame([tft_result])

res_df.to_csv(results_path, index=False)
print('Results saved to model_results.csv:')
print(res_df.to_string(index=False))
print()
print('=' * 50)
print('NOTEBOOK 3 COMPLETE (Kaggle)')
print('=' * 50)
print(f'TFT MAE:  {mae:.4f} °C')
print(f'TFT RMSE: {rmse:.4f} °C')
print(f'TFT R²:   {r2:.4f}')
print()
print('📦 To persist outputs across sessions:')
print('   1. Click "Save Version" (top right) — saves everything in /kaggle/working/')
print('   2. After save, the tft_best.ckpt is available as a notebook output')
print('   3. For Notebook 4: create a new Kaggle Dataset from this output')
print()
print('Next: 04_patchtst_model.ipynb — PatchTST')